# Module 3 — 02: Preprocessing — Heart Disease Datasets

This notebook preprocesses three heart-disease-related datasets that are already available locally under `data/raw/`: cleaning the target variable, handling missing/invalid values, encoding categorical features, and scaling numeric features, for each dataset in turn.

## Dataset 1: Personal Key Indicators of Heart Disease (2022)

### 1. Load Data

In [1]:
import pandas as pd

data_path = '../data/raw/dataset1/heart_2022_with_nans.csv'
df = pd.read_csv(data_path)
print('Loaded file:', data_path)
print('Shape:', df.shape)
df.head()

Loaded file: ../data/raw/dataset1/heart_2022_with_nans.csv
Shape: (445132, 40)


,State,Sex,GeneralHealth,PhysicalHealthDays,MentalHealthDays,LastCheckupTime,PhysicalActivities,SleepHours,RemovedTeeth,HadHeartAttack,...,HeightInMeters,WeightInKilograms,BMI,AlcoholDrinkers,HIVTesting,FluVaxLast12,PneumoVaxEver,TetanusLast10Tdap,HighRiskLastYear,CovidPos
0,Alabama,Female,Very good,0.0,0.0,Within past year (anytime less than 12 months ...,No,8.0,NaN,No,...,NaN,NaN,NaN,No,No,Yes,No,"Yes, received tetanus shot but not sure what type",No,No
1,Alabama,Female,Excellent,0.0,0.0,NaN,No,6.0,NaN,No,...,1.60,68.04,26.57,No,No,No,No,"No, did not receive any tetanus shot in the pa...",No,No
2,Alabama,Female,Very good,2.0,3.0,Within past year (anytime less than 12 months ...,Yes,5.0,NaN,No,...,1.57,63.50,25.61,No,No,No,No,NaN,No,Yes
3,Alabama,Female,Excellent,0.0,0.0,Within past year (anytime less than 12 months ...,Yes,7.0,NaN,No,...,1.65,63.50,23.30,No,No,Yes,Yes,"No, did not receive any tetanus shot in the pa...",No,No
4,Alabama,Female,Fair,2.0,0.0,Within past year (anytime less than 12 months ...,Yes,9.0,NaN,No,...,1.57,53.98,21.77,Yes,No,No,Yes,"No, did not receive any tetanus shot in the pa...",No,No


**Result:** The dataset is loaded directly from the local `data/raw/dataset1/` folder — no download step is needed. We load **`heart_2022_with_nans.csv`** on purpose, since it lets us demonstrate a realistic EDA and preprocessing workflow that includes handling missing values, rather than starting from an already-cleaned file. The loaded dataframe has **445,132 rows and 40 columns**, matching the official 2022 BRFSS update of this dataset.

### 2. Preprocessing

This section prepares the raw data for machine learning: cleaning the target variable, handling missing values, encoding categorical features, and scaling numeric features.

In [2]:
df_clean = df.copy()
before_rows = len(df_clean)
df_clean = df_clean.dropna(subset=['HadHeartAttack']).reset_index(drop=True)
after_rows = len(df_clean)
print(f'Rows before dropping missing target: {before_rows}')
print(f'Rows after dropping missing target: {after_rows}')
print(f'Rows dropped: {before_rows - after_rows}')
df_clean['HadHeartAttack'] = df_clean['HadHeartAttack'].map({'Yes': 1, 'No': 0})
print(df_clean['HadHeartAttack'].value_counts())

Rows before dropping missing target: 445132
Rows after dropping missing target: 442067
Rows dropped: 3065
HadHeartAttack
0    416959
1     25108
Name: count, dtype: int64


**Result:** The 3,065 rows (0.69%) with a missing target value were removed, since imputing a label is not appropriate for a classification target — this leaves 442,067 usable rows. The target was also encoded as a binary integer (Yes=1, No=0) for use in ML models. The resulting class distribution is highly imbalanced: 416,959 negatives (94.32%) vs. 25,108 positives (5.68%), which is important to account for later (e.g. with class weighting or resampling) when training a classifier.

In [3]:
numeric_cols = ['PhysicalHealthDays', 'MentalHealthDays', 'SleepHours', 'HeightInMeters', 'WeightInKilograms', 'BMI']

In [4]:
df_clean[numeric_cols] = df_clean[numeric_cols].fillna(df_clean[numeric_cols].median())
categorical_cols = [c for c in df_clean.columns if c not in numeric_cols and c != 'HadHeartAttack']
mode_values = df_clean[categorical_cols].mode().iloc[0]
df_clean[categorical_cols] = df_clean[categorical_cols].fillna(mode_values)
print('Number of numeric columns imputed (median):', len(numeric_cols))
print('Number of categorical columns imputed (mode):', len(categorical_cols))
print('Total missing values remaining:', df_clean.isna().sum().sum())

Number of numeric columns imputed (median): 6
Number of categorical columns imputed (mode): 33
Total missing values remaining: 0


**Result:** All 6 numeric columns were imputed using the column median (robust to outliers), and all 33 categorical columns were imputed using the column mode (most frequent category). After imputation, the dataset has 0 remaining missing values across all 442,067 rows, so the data is now fully complete and ready for encoding.

In [5]:
binary_cols = [c for c in categorical_cols if df_clean[c].nunique() == 2]
multi_cols = [c for c in categorical_cols if df_clean[c].nunique() > 2]
print('Number of binary categorical columns:', len(binary_cols))
print('Number of multi-category categorical columns:', len(multi_cols))
print(df_clean[categorical_cols].nunique().sort_values().to_string())

Number of binary categorical columns: 22
Number of multi-category categorical columns: 11
Sex                           2
HadStroke                     2
HadAngina                     2
PhysicalActivities            2
HadDepressiveDisorder         2
HadCOPD                       2
HadSkinCancer                 2
HadAsthma                     2
HadKidneyDisease              2
HadArthritis                  2
DeafOrHardOfHearing           2
DifficultyErrands             2
DifficultyDressingBathing     2
DifficultyWalking             2
DifficultyConcentrating       2
BlindOrVisionDifficulty       2
HighRiskLastYear              2
FluVaxLast12                  2
PneumoVaxEver                 2
HIVTesting                    2
AlcoholDrinkers               2
ChestScan                     2
CovidPos                      3
LastCheckupTime               4
ECigaretteUsage               4
TetanusLast10Tdap             4
HadDiabetes                   4
SmokerStatus                  4
RemovedTeeth  

**Result:** Of the 33 categorical columns, 22 are binary (mostly Yes/No health-history flags, e.g. HadStroke, HadAngina, AlcoholDrinkers) and 11 have 3–54 categories. The multi-category columns range from CovidPos (3 categories) up to State (54 categories, i.e. all US states/territories in the survey). Binary columns will be label-encoded to 0/1, and the 11 multi-category columns will be one-hot encoded, since they have no inherent order (except AgeCategory and GeneralHealth, which are ordinal but still small enough to one-hot encode safely).

In [6]:
df_clean[binary_cols] = df_clean[binary_cols].apply(lambda col: pd.factorize(col)[0])
df_clean = pd.get_dummies(df_clean, columns=multi_cols, drop_first=False)
print('Shape before one-hot encoding: (442067, 40)')
print('Shape after one-hot encoding:', df_clean.shape)
print('Number of columns added by one-hot encoding:', df_clean.shape[1] - 40)

Shape before one-hot encoding: (442067, 40)
Shape after one-hot encoding: (442067, 133)
Number of columns added by one-hot encoding: 93


**Result:** The 22 binary categorical columns were label-encoded to integers 0/1, and the 11 multi-category columns were one-hot encoded, expanding the dataset from 40 to 133 columns (93 new dummy columns). This is a large but manageable increase, driven mostly by the State column (54 categories). One-hot encoding avoids implying a false numeric order between unrelated categories (e.g. between different states or races), which is important for most ML algorithms.

In [7]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
df_clean[numeric_cols] = scaler.fit_transform(df_clean[numeric_cols])
print(df_clean[numeric_cols].describe().round(2).to_string())

       PhysicalHealthDays  MentalHealthDays  SleepHours  HeightInMeters  WeightInKilograms        BMI
count           442067.00         442067.00   442067.00       442067.00          442067.00  442067.00
mean                -0.00             -0.00        0.00            0.00              -0.00      -0.00
std                  1.00              1.00        1.00            1.00               1.00       1.00
min                 -0.49             -0.52       -4.05           -7.64              -2.95      -2.65
25%                 -0.49             -0.52       -0.69           -0.70              -0.73      -0.65
50%                 -0.49             -0.52       -0.02           -0.02              -0.10      -0.16
75%                 -0.14             -0.03        0.66            0.75               0.43       0.42
max                  3.00              3.09       11.41            6.82              10.27      11.50


**Result:** The 6 numeric columns were standardized using StandardScaler, so each now has mean ≈ 0 and standard deviation = 1. This puts all numeric features on a comparable scale, which is important for distance-based or gradient-based ML algorithms (e.g. logistic regression, KNN, SVM, neural networks) that would otherwise be dominated by features with larger raw ranges (like WeightInKilograms vs. SleepHours). Note the wide max values for some scaled features (e.g. HeightInMeters max = 6.82, PhysicalHealthDays max = 3.00), reflecting the outliers seen earlier in the EDA step — these are expected and not an error.

In [8]:
print('Final preprocessed dataset shape:', df_clean.shape)
print('Target distribution (%):')
print((df_clean['HadHeartAttack'].value_counts(normalize=True) * 100).round(2))
df_clean.head()

Final preprocessed dataset shape: (442067, 133)
Target distribution (%):
HadHeartAttack
0    94.32
1     5.68
Name: proportion, dtype: float64


,Sex,PhysicalHealthDays,MentalHealthDays,PhysicalActivities,SleepHours,HadHeartAttack,HadAngina,HadStroke,HadAsthma,HadSkinCancer,...,AgeCategory_Age 70 to 74,AgeCategory_Age 75 to 79,AgeCategory_Age 80 or older,"TetanusLast10Tdap_No, did not receive any tetanus shot in the past 10 years","TetanusLast10Tdap_Yes, received Tdap","TetanusLast10Tdap_Yes, received tetanus shot but not sure what type","TetanusLast10Tdap_Yes, received tetanus shot, but not Tdap",CovidPos_No,CovidPos_Tested positive using home test without a health professional,CovidPos_Yes
0,0,-0.491530,-0.515488,0,0.656772,0,0,0,0,0,...,False,False,True,False,False,True,False,True,False,False
1,0,-0.491530,-0.515488,0,-0.687687,0,0,0,0,1,...,False,False,True,True,False,False,False,True,False,False
2,0,-0.258523,-0.154541,1,-1.359916,0,0,0,0,1,...,False,False,False,True,False,False,False,False,False,True
3,0,-0.491530,-0.515488,1,-0.015457,0,0,0,1,0,...,False,False,False,True,False,False,False,True,False,False
4,0,-0.258523,-0.515488,1,1.329001,0,0,0,0,0,...,False,False,False,True,False,False,False,True,False,False


**Result:** The final preprocessed dataset has 442,067 rows and 133 columns, with 0 missing values, a binary numeric target (HadHeartAttack), all categorical features encoded (label encoding for binary columns, one-hot encoding for multi-category columns), and all 6 numeric features standardized. The target remains imbalanced (94.32% No vs. 5.68% Yes), which was preserved intentionally rather than artificially rebalanced, since resampling should typically be applied only to the training split (after a train/test split) to avoid data leakage. This dataset is now ready to be split into training and test sets and used to train a classification model for heart attack risk prediction.

## Dataset 2: Heart Failure Prediction

### 1. Load Data

In [9]:
data_path = '../data/raw/dataset2/heart.csv'
df2 = pd.read_csv(data_path)
print('Loaded file:', data_path)
print('Shape:', df2.shape)
df2.head()

Loaded file: ../data/raw/dataset2/heart.csv
Shape: (918, 12)


,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


**Result:** Loaded from the local `data/raw/dataset2/` folder. **918 rows and 12 columns**, target `HeartDisease` already encoded as 0/1.

### 2. Preprocessing

Dataset 2's target is already a clean binary integer, and there are no NaN values — but `Cholesterol` and `RestingBP` use `0` as a placeholder for a measurement that wasn't taken, so those are converted to genuine missing values before imputation.

In [10]:
df2_clean = df2.copy()
print('HeartDisease dtype:', df2_clean['HeartDisease'].dtype)
print('Missing target values:', df2_clean['HeartDisease'].isna().sum())
print(df2_clean['HeartDisease'].value_counts())

HeartDisease dtype: int64
Missing target values: 0
HeartDisease
1    508
0    410
Name: count, dtype: int64


**Result:** `HeartDisease` is already a numeric 0/1 column (`int64`) with no missing values, so no dropping or Yes/No mapping is needed here — unlike Dataset 1. The class split is 508 (55.34%) positive vs. 410 (44.66%) negative.

In [11]:
import numpy as np

zero_as_missing_cols = ['Cholesterol', 'RestingBP']
for col in zero_as_missing_cols:
    n_zero = (df2_clean[col] == 0).sum()
    df2_clean[col] = df2_clean[col].replace(0, np.nan)
    print(f'{col}: converted {n_zero} zero-value rows to NaN')
print()
print(df2_clean[zero_as_missing_cols].isna().sum())

Cholesterol: converted 172 zero-value rows to NaN
RestingBP: converted 1 zero-value rows to NaN

Cholesterol    172
RestingBP        1
dtype: int64


**Result:** 172 zero-value `Cholesterol` readings and 1 zero-value `RestingBP` reading were converted to `NaN`, since a true reading of 0 for either is not physiologically possible. These are now genuine missing values that the imputation step below will fill in with the column median, the same way Dataset 1's naturally-occurring NaNs are handled.

In [12]:
numeric_cols2 = ['Age', 'RestingBP', 'Cholesterol', 'FastingBS', 'MaxHR', 'Oldpeak']

In [13]:
df2_clean[numeric_cols2] = df2_clean[numeric_cols2].fillna(df2_clean[numeric_cols2].median())
categorical_cols2 = [c for c in df2_clean.columns if c not in numeric_cols2 and c != 'HeartDisease']
mode_values2 = df2_clean[categorical_cols2].mode().iloc[0]
df2_clean[categorical_cols2] = df2_clean[categorical_cols2].fillna(mode_values2)
print('Number of numeric columns imputed (median):', len(numeric_cols2))
print('Number of categorical columns imputed (mode):', len(categorical_cols2))
print('Total missing values remaining:', df2_clean.isna().sum().sum())

Number of numeric columns imputed (median): 6
Number of categorical columns imputed (mode): 5
Total missing values remaining: 0


**Result:** All 6 numeric columns (including the now-missing `Cholesterol`/`RestingBP` entries) were imputed using the column median, and the 5 categorical columns were imputed using the mode (though none of them actually had missing values to begin with). After imputation, the dataset has 0 remaining missing values across all 918 rows.

In [14]:
binary_cols2 = [c for c in categorical_cols2 if df2_clean[c].nunique() == 2]
multi_cols2 = [c for c in categorical_cols2 if df2_clean[c].nunique() > 2]
print('Number of binary categorical columns:', len(binary_cols2))
print('Number of multi-category categorical columns:', len(multi_cols2))
print(df2_clean[categorical_cols2].nunique().sort_values().to_string())

Number of binary categorical columns: 2
Number of multi-category categorical columns: 3
Sex               2
ExerciseAngina    2
RestingECG        3
ST_Slope          3
ChestPainType     4


**Result:** Of the 5 categorical columns, 2 are binary (`Sex`, `ExerciseAngina`) and 3 have 3–4 categories (`RestingECG`, `ST_Slope`, `ChestPainType`). Binary columns will be label-encoded to 0/1, and the 3 multi-category columns will be one-hot encoded.

In [15]:
df2_clean[binary_cols2] = df2_clean[binary_cols2].apply(lambda col: pd.factorize(col)[0])
df2_clean = pd.get_dummies(df2_clean, columns=multi_cols2, drop_first=False)
print('Shape before one-hot encoding: (918, 12)')
print('Shape after one-hot encoding:', df2_clean.shape)
print('Number of columns added by one-hot encoding:', df2_clean.shape[1] - 12)

Shape before one-hot encoding: (918, 12)
Shape after one-hot encoding: (918, 19)
Number of columns added by one-hot encoding: 7


**Result:** The 2 binary columns were label-encoded to 0/1, and the 3 multi-category columns (10 categories combined: 4 ChestPainType + 3 RestingECG + 3 ST_Slope) were one-hot encoded, expanding the dataset from 12 to 19 columns (7 net new dummy columns).

In [16]:
scaler2 = StandardScaler()
df2_clean[numeric_cols2] = scaler2.fit_transform(df2_clean[numeric_cols2])
print(df2_clean[numeric_cols2].describe().round(2).to_string())

          Age  RestingBP  Cholesterol  FastingBS   MaxHR  Oldpeak
count  918.00     918.00       918.00     918.00  918.00   918.00
mean    -0.00       0.00         0.00      -0.00    0.00     0.00
std      1.00       1.00         1.00       1.00    1.00     1.00
min     -2.71      -2.92        -2.96      -0.55   -3.02    -3.27
25%     -0.69      -0.70        -0.55      -0.55   -0.66    -0.83
50%      0.05      -0.14        -0.12      -0.55    0.05    -0.27
75%      0.69       0.42         0.45      -0.55    0.75     0.57
max      2.49       3.75         6.74       1.81    2.56     4.98


**Result:** The 6 numeric columns were standardized so each now has mean ≈ 0 and standard deviation = 1, putting `Age`, blood pressure, cholesterol, and heart-rate measurements on a comparable scale despite their very different raw units and ranges.

In [17]:
print('Final preprocessed dataset shape:', df2_clean.shape)
print('Target distribution (%):')
print((df2_clean['HeartDisease'].value_counts(normalize=True) * 100).round(2))
df2_clean.head()

Final preprocessed dataset shape: (918, 19)
Target distribution (%):
HeartDisease
1    55.34
0    44.66
Name: proportion, dtype: float64


,Age,Sex,RestingBP,Cholesterol,FastingBS,MaxHR,ExerciseAngina,Oldpeak,HeartDisease,ChestPainType_ASY,ChestPainType_ATA,ChestPainType_NAP,ChestPainType_TA,RestingECG_LVH,RestingECG_Normal,RestingECG_ST,ST_Slope_Down,ST_Slope_Flat,ST_Slope_Up
0,-1.433140,0,0.415002,0.858035,-0.551341,1.382928,0,-0.832432,0,False,True,False,False,False,True,False,False,False,True
1,-0.478484,1,1.527329,-1.184227,-0.551341,0.754157,0,0.105664,1,False,False,True,False,False,True,False,False,True,False
2,-1.751359,0,-0.141161,0.745617,-0.551341,-1.525138,0,-0.832432,0,False,True,False,False,False,False,True,False,False,True
3,-0.584556,1,0.303769,-0.547191,-0.551341,-1.132156,1,0.574711,1,True,False,False,False,False,True,False,False,True,False
4,0.051881,0,0.971166,-0.903182,-0.551341,-0.581981,0,-0.832432,0,False,False,True,False,False,True,False,False,False,True


**Result:** The final preprocessed dataset has 918 rows and 19 columns, with 0 missing values, a binary numeric target (`HeartDisease`), all categorical features encoded, and all 6 numeric features standardized. The near-balanced target distribution (55.34% positive vs. 44.66% negative) was preserved as-is. This dataset is now ready to be split into training and test sets.

## Dataset 3: Heart Disease Health Indicators (BRFSS 2015)

### 1. Load Data

In [18]:
data_path = '../data/raw/dataset3/heart_disease_health_indicators_BRFSS2015.csv'
df3 = pd.read_csv(data_path)
print('Loaded file:', data_path)
print('Shape:', df3.shape)
df3.head()

Loaded file: ../data/raw/dataset3/heart_disease_health_indicators_BRFSS2015.csv
Shape: (253680, 22)


,HeartDiseaseorAttack,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,Diabetes,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0


**Result:** Loaded from the local `data/raw/dataset3/` folder. **253,680 rows and 22 columns**, all already numerically encoded, with the binary target `HeartDiseaseorAttack`.

### 2. Preprocessing

Dataset 3 has no missing values and an already-numeric target, but **9.42% of its rows are exact duplicates** — these are dropped first, before the target-cleaning step, so they don't distort the class distribution or get split across both the train and test sets.

In [19]:
df3_clean = df3.copy()
before_dup = len(df3_clean)
df3_clean = df3_clean.drop_duplicates().reset_index(drop=True)
after_dup = len(df3_clean)
print(f'Rows before dropping duplicates: {before_dup}')
print(f'Rows after dropping duplicates: {after_dup}')
print(f'Duplicate rows dropped: {before_dup - after_dup}')

Rows before dropping duplicates: 253680
Rows after dropping duplicates: 229781
Duplicate rows dropped: 23899


**Result:** 23,899 exact-duplicate rows (9.42%) were dropped, leaving 229,781 rows. This mirrors the finding from the EDA notebook: with 22 binary/small-discrete-scale features, distinct respondents can easily share an identical answer pattern, so these duplicates are removed to avoid the model over-weighting whichever response pattern happens to repeat.

In [20]:
before_rows3 = len(df3_clean)
df3_clean = df3_clean.dropna(subset=['HeartDiseaseorAttack']).reset_index(drop=True)
after_rows3 = len(df3_clean)
print(f'Rows before dropping missing target: {before_rows3}')
print(f'Rows after dropping missing target: {after_rows3}')
print(f'Rows dropped: {before_rows3 - after_rows3}')
df3_clean['HeartDiseaseorAttack'] = df3_clean['HeartDiseaseorAttack'].astype(int)
print(df3_clean['HeartDiseaseorAttack'].value_counts())

Rows before dropping missing target: 229781
Rows after dropping missing target: 229781
Rows dropped: 0
HeartDiseaseorAttack
0    206064
1     23717
Name: count, dtype: int64


**Result:** `HeartDiseaseorAttack` had no missing values, so no rows were dropped in this step (229,781 rows before and after) — it only converts the target from `float64` to a clean `int` 0/1. After the earlier duplicate removal, the class distribution shifted slightly to 206,064 negatives (89.68%) vs. 23,717 positives (10.32%), still heavily imbalanced but marginally less so than the raw 90.58%/9.42% split, since a disproportionate share of the dropped duplicate rows were negative.

In [21]:
numeric_cols3 = ['BMI', 'MentHlth', 'PhysHlth', 'GenHlth', 'Age', 'Education', 'Income']

In [22]:
df3_clean[numeric_cols3] = df3_clean[numeric_cols3].fillna(df3_clean[numeric_cols3].median())
categorical_cols3 = [c for c in df3_clean.columns if c not in numeric_cols3 and c != 'HeartDiseaseorAttack']
mode_values3 = df3_clean[categorical_cols3].mode().iloc[0]
df3_clean[categorical_cols3] = df3_clean[categorical_cols3].fillna(mode_values3)
print('Number of numeric columns imputed (median):', len(numeric_cols3))
print('Number of categorical columns imputed (mode):', len(categorical_cols3))
print('Total missing values remaining:', df3_clean.isna().sum().sum())

Number of numeric columns imputed (median): 7
Number of categorical columns imputed (mode): 14
Total missing values remaining: 0


**Result:** The imputation step runs over all 7 numeric and 14 categorical columns for consistency with the other two datasets, but since Dataset 3 had zero missing values to begin with, nothing actually changes here — the total missing-value count stays at 0.

In [23]:
binary_cols3 = [c for c in categorical_cols3 if df3_clean[c].nunique() == 2]
multi_cols3 = [c for c in categorical_cols3 if df3_clean[c].nunique() > 2]
print('Number of binary categorical columns:', len(binary_cols3))
print('Number of multi-category categorical columns:', len(multi_cols3))
print(df3_clean[categorical_cols3].nunique().sort_values().to_string())

Number of binary categorical columns: 13
Number of multi-category categorical columns: 1
HighBP               2
HighChol             2
CholCheck            2
Smoker               2
Stroke               2
PhysActivity         2
Fruits               2
Veggies              2
DiffWalk             2
HvyAlcoholConsump    2
AnyHealthcare        2
NoDocbcCost          2
Sex                  2
Diabetes             3


**Result:** Of the 14 non-numeric feature columns, 13 are binary flags (`HighBP`, `HighChol`, `CholCheck`, `Smoker`, `Stroke`, `PhysActivity`, `Fruits`, `Veggies`, `HvyAlcoholConsump`, `AnyHealthcare`, `NoDocbcCost`, `DiffWalk`, `Sex`) and only 1 is multi-category: `Diabetes`, which has 3 levels (0 = no diabetes, 1 = pre-diabetes, 2 = diabetes).

In [24]:
df3_clean[binary_cols3] = df3_clean[binary_cols3].apply(lambda col: pd.factorize(col)[0])
df3_clean = pd.get_dummies(df3_clean, columns=multi_cols3, drop_first=False)
print('Shape before one-hot encoding: (229781, 22)')
print('Shape after one-hot encoding:', df3_clean.shape)
print('Number of columns added by one-hot encoding:', df3_clean.shape[1] - 22)

Shape before one-hot encoding: (229781, 22)
Shape after one-hot encoding: (229781, 24)
Number of columns added by one-hot encoding: 2


**Result:** The 13 binary columns were label-encoded to 0/1 (a no-op for most, since they were already 0.0/1.0 floats), and the single 3-category `Diabetes` column was one-hot encoded into 3 dummy columns, expanding the dataset from 22 to 24 columns (2 net new columns).

In [25]:
scaler3 = StandardScaler()
df3_clean[numeric_cols3] = scaler3.fit_transform(df3_clean[numeric_cols3])
print(df3_clean[numeric_cols3].describe().round(2).to_string())

             BMI   MentHlth   PhysHlth    GenHlth        Age  Education     Income
count  229781.00  229781.00  229781.00  229781.00  229781.00  229781.00  229781.00
mean       -0.00       0.00      -0.00       0.00      -0.00      -0.00      -0.00
std         1.00       1.00       1.00       1.00       1.00       1.00       1.00
min        -2.46      -0.45      -0.52      -1.50      -2.29      -4.01      -2.34
25%        -0.69      -0.45      -0.52      -0.56      -0.67      -0.99      -0.43
50%        -0.25      -0.45      -0.52       0.37      -0.03       0.02       0.05
75%         0.49      -0.20      -0.07       0.37       0.62       1.03       1.01
max        10.21       3.43       2.80       2.25       1.59       1.03       1.01


**Result:** The 7 numeric/ordinal columns were standardized so each now has mean ≈ 0 and standard deviation = 1, putting `BMI` and the various small ordinal scales (`GenHlth`, `Age`, `Education`, `Income`) on the same footing rather than letting the wider-range `BMI` dominate distance-based algorithms.

In [26]:
print('Final preprocessed dataset shape:', df3_clean.shape)
print('Target distribution (%):')
print((df3_clean['HeartDiseaseorAttack'].value_counts(normalize=True) * 100).round(2))
df3_clean.head()

Final preprocessed dataset shape: (229781, 24)
Target distribution (%):
HeartDiseaseorAttack
0    89.68
1    10.32
Name: proportion, dtype: float64


,HeartDiseaseorAttack,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,PhysActivity,Fruits,Veggies,...,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income,Diabetes_0.0,Diabetes_1.0,Diabetes_2.0
0,0,0,0,0,1.667220,0,0,0,0,0,...,1.879074,1.141300,0,0,0.295241,-0.987588,-1.381324,True,False,False
1,0,1,1,1,-0.543101,0,0,1,0,1,...,-0.454434,-0.516791,1,0,-0.351213,1.026729,-2.337131,True,False,False
2,0,0,0,0,-0.101037,1,0,0,1,1,...,3.434746,2.799391,0,0,0.295241,-0.987588,1.008193,True,False,False
3,0,0,1,0,-0.248391,1,0,1,1,0,...,-0.454434,-0.516791,1,0,0.941695,-1.994746,0.052387,True,False,False
4,0,0,0,0,-0.690456,1,0,1,1,0,...,-0.065516,-0.516791,1,0,0.941695,0.019571,-0.903420,True,False,False


**Result:** The final preprocessed dataset has 229,781 rows and 24 columns, with 0 missing values, a binary numeric target (`HeartDiseaseorAttack`), all categorical features encoded (label encoding for the 13 binary flags, one-hot encoding for `Diabetes`), and all 7 numeric/ordinal features standardized. The target remains imbalanced (89.68% No vs. 10.32% Yes), preserved as-is rather than resampled at this stage. This dataset is now ready to be split into training and test sets.